# Legitimate Academic Announcements vs. Spam: TF-IDF vs. Sentence Embeddings
**NLP Course Project - Option 1 (Spam Detection), Ling-Spam dataset**

**Runtime:** Google Colab (also runs on local Jupyter). Run all cells in order from a fresh session.

**Experimental question:** Which method more reliably distinguishes legitimate academic announcements containing promotional vocabulary from genuine spam: TF-IDF + logistic regression, or pretrained sentence embeddings (all-MiniLM-L6-v2) + logistic regression?

| Section | Owner |
|---|---|
| Setup, Data, Method 1 (TF-IDF) | Abrar |
| Method 2 (sentence embeddings), Try it for Method 2, reproducibility check | Jana |
| Results (challenge subset, metrics, false-positive analysis, error analysis) | Zahra |


## 1. Setup

In [1]:
import sys, os, subprocess, random
import numpy as np

# --- Colab: install pinned versions and fetch the project files ---------------
# Versions below are the ones this project was developed and tested with.
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "scikit-learn==1.9.1", "pandas==3.0.5", "numpy==2.5.3",
                    "scipy==1.18.1", "joblib==1.6.0"], check=True)
    # TODO (Abrar): replace with the real GitHub repo URL before submitting
    REPO_URL = "https://github.com/AbrarAlmutawa/Distinguishing-Legitimate-Announcements-from-Spam-TF-IDF-vs.-Sentence-Embeddings.git"
    REPO_DIR = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.exists("data_utils.py"):
        if not os.path.exists(REPO_DIR):
            subprocess.run(["git", "clone", REPO_URL], check=True)
        os.chdir(REPO_DIR)   # all later paths are relative to the repo root

# --- Reproducibility -----------------------------------------------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED)

import sklearn, pandas as pd, scipy, joblib
print("python", sys.version.split()[0], "| sklearn", sklearn.__version__,
      "| pandas", pd.__version__, "| numpy", np.__version__)
print("working directory:", os.getcwd())
assert os.path.exists("data/messages.csv"), "data/messages.csv not found - run from the repo root"

python 3.13.16 | sklearn 1.9.1 | pandas 3.0.5 | numpy 2.4.6
working directory: /content/Distinguishing-Legitimate-Announcements-from-Spam-TF-IDF-vs.-Sentence-Embeddings


## 2. Data
**Source:** Ling-Spam corpus, Kaggle version (`messages.csv`). Columns: `subject`, `message`, `label` (0 = legitimate, 1 = spam).

Steps (feedback points 4 and 5):
1. Verify the format: separate `subject` and `message` fields exist, so the model input is `subject + " " + message`.
2. Normalise (lowercase, collapse whitespace) and remove exact duplicates.
3. Group near-duplicates (TF-IDF cosine similarity > 0.9) with connected components.
4. Stratified 70/15/15 split in which every near-duplicate group stays inside one partition.
5. Programmatically verify that no group or text crosses partitions.

In [2]:
import json
import data_utils as du

raw = du.load_raw("data/messages.csv")
print("columns:", list(raw.columns[:3]), "| rows:", len(raw))
print("class counts (0 = legitimate, 1 = spam):", raw["label"].value_counts().to_dict())
print("messages with an empty subject:", int((raw["subject"].str.strip() == "").sum()))
raw[["subject", "message", "label"]].head(3)

columns: ['subject', 'message', 'label'] | rows: 2893
class counts (0 = legitimate, 1 = spam): {0: 2412, 1: 481}
messages with an empty subject: 62


,subject,message,label
0,job posting - apple-iss research center,content - length : 3386 apple-iss research cen...,0
1,,"lang classification grimes , joseph e . and ba...",0
2,query : letter frequencies for text identifica...,i am posting this inquiry for sergei atamas ( ...,0


In [3]:
clean, exact_info = du.drop_exact_duplicates(raw)
print(exact_info)

clean["group"], near_info = du.near_duplicate_groups(clean, threshold=0.9)
print(near_info)

clean["split"] = du.grouped_stratified_split(clean)
train, dev, test = (clean[clean["split"] == s].reset_index(drop=True) for s in ("train", "dev", "test"))
du.check_no_leakage(train, dev, test)
print("leakage check passed: no group or normalised text shared across partitions")
pd.DataFrame(du.summarise_splits(train, dev, test)).T

{'exact_duplicates_removed': 17, 'removed_spam': 13, 'removed_legit': 4, 'exact_duplicate_texts_with_conflicting_labels': 0}
{'near_dup_threshold': 0.9, 'near_dup_groups_with_2plus_messages': 141, 'messages_in_near_dup_groups': 319, 'largest_group_size': 5}
leakage check passed: no group or normalised text shared across partitions


,n,spam,legit,spam_share
train,2014.0,326.0,1688.0,0.1619
dev,430.0,70.0,360.0,0.1628
test,432.0,72.0,360.0,0.1667


In [4]:
# Save the partitions (the committed copies in data/ are identical because everything is seeded)
cols = ["subject", "message", "text", "label", "group"]
for name, part in (("train", train), ("dev", dev), ("test", test)):
    part[cols].to_csv(f"data/{name}.csv", index=False)

## 3. Method 1 - TF-IDF + Logistic Regression (traditional)
TF-IDF turns each message into a weighted bag of words; logistic regression classifies it. Both come from scikit-learn (no pretrained weights).

Only the **training** set is used to fit and only the **development** set to choose settings: n-gram range, `min_df`, `C` and the decision threshold. The test set is not loaded in this section.

In [5]:
import baseline_tfidf as m1

tfidf_model, tfidf_cfg, grid = m1.run(save=True, verbose=False)
print("top of the dev grid (24 combinations tried):")
display(grid.head(8))
print("selected settings:", {k: tfidf_cfg[k] for k in ("ngram_range", "min_df", "C", "threshold")})
print(f"DEV macro F1 {tfidf_cfg['dev_macro_f1']} | DEV accuracy {tfidf_cfg['dev_accuracy']} | "
      f"DEV false positives {tfidf_cfg['dev_false_positives']} | DEV false negatives {tfidf_cfg['dev_false_negatives']}")

top of the dev grid (24 combinations tried):


,ngram_range,min_df,C,dev_macro_f1,dev_accuracy
2,"(1, 1)",1,10.0,0.995709,0.997674
3,"(1, 1)",1,100.0,0.995709,0.997674
7,"(1, 1)",2,100.0,0.995709,0.997674
11,"(1, 1)",3,100.0,0.995709,0.997674
6,"(1, 1)",2,10.0,0.991369,0.995349
10,"(1, 1)",3,10.0,0.991369,0.995349
5,"(1, 1)",2,1.0,0.986976,0.993023
9,"(1, 1)",3,1.0,0.986976,0.993023


selected settings: {'ngram_range': [1, 1], 'min_df': 1, 'C': 10.0, 'threshold': 0.5}
DEV macro F1 0.9957 | DEV accuracy 0.9977 | DEV false positives 0 | DEV false negatives 1


Saved to `models/tfidf_lr.joblib` (pipeline) and `models/tfidf_lr_config.json` (settings and threshold) so the **Try it** section can load the model without retraining.

## 4. Method 2 - Sentence embeddings + Logistic Regression (pretrained)
**Owner: Jana.** Code: `embed_sbert.py`.

1. **Encoder (pretrained, frozen):** `sentence-transformers/all-MiniLM-L6-v2` (Reimers and Gurevych, 2019) maps each message (`subject + " " + message`, same input as Method 1) to a 384-dimensional, L2-normalised sentence embedding. We do not fine-tune it; the weights are downloaded from Hugging Face on first use.
2. **Classifier:** logistic regression (scikit-learn) with balanced class weights, trained on the **training** embeddings only.
3. **Tuning on dev only:** `C` from {0.01, 0.1, 1, 10, 100} is picked by dev macro F1, then the decision threshold is picked on dev with the same rule as Method 1 (ties go to the higher threshold, which means fewer false positives).
4. **Saved for reuse:** `models/sbert_lr.joblib` (classifier) and `models/sbert_lr_config.json` (settings, threshold, encoder revision). Embeddings are cached in `data/emb_cache/`.

The test set is not loaded in this section.

**Limitation:** the encoder reads at most 256 word-piece tokens, so a long email is represented by its subject and the start of its body.


In [6]:
# Colab: install the Method 2 packages (pinned; same versions as requirements.txt)
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "sentence-transformers==6.3.0", "transformers==5.24.1"], check=True)

import torch, sentence_transformers, transformers
torch.manual_seed(SEED)
print("sentence-transformers", sentence_transformers.__version__, "| transformers", transformers.__version__,
      "| torch", torch.__version__, "| device:", "cuda" if torch.cuda.is_available() else "cpu")

import embed_sbert as m2

sbert_model, sbert_cfg, sbert_grid = m2.run(save=True, verbose=False)
print(f"dev grid ({sbert_cfg['grid_combinations_tried']} values of C tried):")
display(sbert_grid)
print("selected settings:", {k: sbert_cfg[k] for k in ("C", "threshold")},
      "| encoder revision:", sbert_cfg["encoder_revision"])
print(f"DEV macro F1 {sbert_cfg['dev_macro_f1']} | DEV accuracy {sbert_cfg['dev_accuracy']} | "
      f"DEV false positives {sbert_cfg['dev_false_positives']} | DEV false negatives {sbert_cfg['dev_false_negatives']}")


sentence-transformers 6.3.0 | transformers 5.24.1 | torch 2.14.1+cu130 | device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (9 outputs left out)

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

dev grid (5 values of C tried):


,C,dev_macro_f1,dev_accuracy
3,10.00,0.987128,0.993023
4,100.00,0.987128,0.993023
2,1.00,0.983130,0.990698
0,0.01,0.975250,0.986047
1,0.10,0.975250,0.986047


selected settings: {'C': 10.0, 'threshold': 0.45} | encoder revision: None
DEV macro F1 0.9873 | DEV accuracy 0.993 | DEV false positives 2 | DEV false negatives 1


## 5. Results
**Owner: Zahra.** Run the test set **once**, after both models are final.
- Accuracy, macro F1 and confusion matrices on the identical test set.
- Challenge subset: legitimate test messages passing the keyword filter, then verified by hand as academic announcements. Document the rules and the count, and fix the subset before looking at predictions.
- Number of false positives and False Positive Rate (FPR) on the subset, for both models.
- Error analysis (3+ wrong predictions, or all of them plus difficult correct cases) and limitations (small subset, so FPR is noisy).

In [7]:
# TODO (Zahra): evaluation code goes here

## 6. Try it
Type any email (subject and body) below. The saved model is loaded from disk; nothing is retrained. Empty input returns a helpful message.

In [8]:
import joblib

_tfidf = joblib.load("models/tfidf_lr.joblib")
_cfg = json.load(open("models/tfidf_lr_config.json"))
LABELS = {0: "Not spam", 1: "Spam"}

def predict_tfidf(subject="", body=""):
    """Return (label, confidence) for one email using the saved TF-IDF + LR model."""
    text = du.build_input_text(subject, body)
    if not text.strip():
        return None, None
    p_spam = float(_tfidf.predict_proba([text])[0, 1])
    pred = int(p_spam >= _cfg["threshold"])
    return LABELS[pred], (p_spam if pred == 1 else 1 - p_spam)

def try_it(subject="", body=""):
    label, conf = predict_tfidf(subject, body)
    if label is None:
        print("Please enter a subject and/or message body - the input was empty.")
    else:
        print(f"TF-IDF + LR  ->  {label}  (confidence {conf:.1%})")

try_it("Call for papers: workshop on computational syntax",
       "Submission deadline 15 March. Registration is free for students.")
try_it("", "")

TF-IDF + LR  ->  Not spam  (confidence 96.3%)
Please enter a subject and/or message body - the input was empty.


In [9]:
# Interactive version: edit the two strings and re-run this cell
try_it(subject="Earn $5,000 a week from home",
       body="Click here now to claim your free bonus. Limited time opportunity!")

TF-IDF + LR  ->  Spam  (confidence 99.6%)


### Try it with Method 2 (sentence embeddings) and both models side by side
`predict_sbert` loads the saved classifier and the pretrained encoder; nothing is retrained. `try_it_both` shows the two models' predictions for the same email.

In [10]:
_sbert_clf, _sbert_cfg = m2.load_saved()   # saved classifier + dev-chosen threshold

def predict_sbert(subject="", body=""):
    """Return (label, confidence) for one email using the saved embedding + LR model."""
    text = du.build_input_text(subject, body)
    if not text.strip():
        return None, None
    p_spam = float(m2.predict_proba_texts([text], _sbert_clf, _sbert_cfg)[0])
    pred = int(p_spam >= _sbert_cfg["threshold"])
    return LABELS[pred], (p_spam if pred == 1 else 1 - p_spam)

def try_it_both(subject="", body=""):
    """Print both models' predictions for one email; handles empty input."""
    if not du.build_input_text(subject, body).strip():
        print("Please enter a subject and/or message body - the input was empty.")
        return
    for name, fn in (("TF-IDF + LR       ", predict_tfidf), ("Embeddings + LR   ", predict_sbert)):
        label, conf = fn(subject, body)
        print(f"{name} ->  {label}  (confidence {conf:.1%})")

try_it_both("Call for papers: workshop on computational syntax",
            "Submission deadline 15 March. Registration is free for students.")
print()
try_it_both("Earn $5,000 a week from home",
            "Click here now to claim your free bonus. Limited time opportunity!")
print()
try_it_both("", "   ")


TF-IDF + LR        ->  Not spam  (confidence 96.3%)
Embeddings + LR    ->  Not spam  (confidence 99.9%)

TF-IDF + LR        ->  Spam  (confidence 99.6%)
Embeddings + LR    ->  Spam  (confidence 99.9%)

Please enter a subject and/or message body - the input was empty.


In [11]:
# Interactive version: edit the two strings and re-run this cell
try_it_both(subject="Free registration for the summer school in computational linguistics",
            body="Register before the deadline to secure your place. Travel grants available for students.")


TF-IDF + LR        ->  Not spam  (confidence 81.6%)
Embeddings + LR    ->  Not spam  (confidence 98.5%)


### Original labelled examples (written by the team - not part of the test scores)
Each member adds two examples here with the label they believe is correct. These are shown separately from the test-set results.

In [12]:
# Abrar's two examples are added after review (see report/original_examples_abrar.md)
original_examples = []   # list of (subject, body, true_label) tuples; each member appends two
for subj, body, truth in original_examples:
    label, conf = predict_tfidf(subj, body)
    print(f"true: {truth:9s} | predicted: {label} ({conf:.1%}) | {subj}")

## 7. Sources
- Ling-Spam corpus: I. Androutsopoulos et al., "An evaluation of naive Bayesian anti-spam filtering," ECML Workshop on Machine Learning in the New Information Age, 2000. Kaggle copy used: `messages.csv` (add the exact Kaggle dataset URL here).
- N. Reimers and I. Gurevych, "Sentence-BERT: Sentence embeddings using Siamese BERT-networks," EMNLP-IJCNLP, 2019.
- Libraries: scikit-learn (TF-IDF, logistic regression, splitting utilities), pandas, numpy, scipy, joblib; `sentence-transformers` and `all-MiniLM-L6-v2` for Method 2.
- AI assistance: Claude (Anthropic) helped draft the data pipeline and baseline code; the team reviewed, ran and verified it.